# 08c exo video prep

Cuts the four episodes out of the root phone video and writes one cam_ext stream per episode.

## 1. Setup

Loads the config, the root timestamps and the time anchor, and checks ffmpeg and the free disk space.

In [1]:
import hashlib
import json
import mmap
import re
import shutil
import subprocess
import time
import zipfile
import zlib
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import av
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from tqdm import tqdm

# flags
FORCE = False
PRESET = "veryfast"
CRF_CANDIDATES = [24, 26, 28, 30, 32]
TARGET_SSIM = 0.98
BITRATE_CAP_MBPS = 8.0
ROOT_VIDEO_NAME = "video_20261011_200844.mp4"
MIN_FREE_GB = 15

# folders
PROJECT_ROOT = Path.cwd().parent
cfg_path = PROJECT_ROOT / "configs" / "config.yaml"
if not cfg_path.exists():
    raise SystemExit("configs\\config.yaml not found in " + str(PROJECT_ROOT) + ". open this notebook from the notebooks folder of the project.")

RAW = PROJECT_ROOT / "raw"
INTERIM = PROJECT_ROOT / "interim"
EPISODES = PROJECT_ROOT / "episodes"
REPORTS = PROJECT_ROOT / "reports"
ROOT_VIDEO = RAW / ROOT_VIDEO_NAME
REPORTS.mkdir(exist_ok=True)


# small helpers
def need(path):
    return FORCE or not Path(path).exists()


def run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")


def read_json(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        return json.load(f)


def save_csv(df, path):
    if need(path):
        df.to_csv(path, index=False)
        print("saved", Path(path).name)
    else:
        print("kept existing", Path(path).name)


def ffprobe_json(args):
    r = run(["ffprobe", "-v", "error", "-of", "json"] + args)
    if r.returncode != 0 or not r.stdout.strip():
        raise RuntimeError("ffprobe failed: " + r.stderr[-300:])
    return json.loads(r.stdout)


iso_re = re.compile(r"^(\d{4})-(\d{2})-(\d{2})[T ](\d{2}):(\d{2}):(\d{2})(?:\.(\d+))?\s*(Z|[+-]\d{2}:?\d{2})?$")


def iso_to_ns(text):
    m = iso_re.match(str(text).strip())
    if m is None:
        raise ValueError("cannot read time " + repr(text))
    y, mo, d, h, mi, s = (int(m.group(i)) for i in range(1, 7))
    frac = (m.group(7) or "0")[:9].ljust(9, "0")
    tz = m.group(8)
    base = datetime(y, mo, d, h, mi, s, tzinfo=timezone.utc)
    ns = int(base.timestamp()) * 10**9 + int(frac)
    if tz and tz != "Z":
        sign = 1 if tz[0] == "+" else -1
        digits = tz[1:].replace(":", "")
        ns -= sign * (int(digits[:2]) * 3600 + int(digits[2:]) * 60) * 10**9
    return ns


def run_lengths(mask):
    lens = []
    cur = 0
    for v in mask:
        if v:
            cur += 1
        elif cur:
            lens.append(cur)
            cur = 0
    if cur:
        lens.append(cur)
    return lens


def real_duration(first, last):
    # seconds covered by frames first..last, up to the start of the next frame
    if last + 1 < len(pts):
        return float(pts[last + 1] - pts[first])
    n = last - first + 1
    return float((pts[last] - pts[first]) * n / max(n - 1, 1))


try:
    with open(cfg_path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    print("config loaded:", list(config.keys())[:8] if isinstance(config, dict) else type(config).__name__)

    ffmpeg_line = run(["ffmpeg", "-version"]).stdout.splitlines()[0]
    ffprobe_line = run(["ffprobe", "-version"]).stdout.splitlines()[0]
    print(ffmpeg_line)
    print(ffprobe_line)

    anchor = read_json(REPORTS / "04_time_anchor.json")
    root_start_utc = anchor["root_start_utc"]
    root_start_ns = iso_to_ns(root_start_utc)
    print("root_start_utc:", root_start_utc)

    npz = np.load(INTERIM / "root_motion_30hz.npz")
    print("npz keys:", npz.files)
    key = "pts_s" if "pts_s" in npz.files else npz.files[0]
    if key != "pts_s":
        print("pts_s not found, using first array:", key)
    pts = np.asarray(npz[key], dtype=np.float64)
    dt = np.diff(pts)
    print("root frames:", len(pts), " duration: %.1f s" % (pts[-1] - pts[0]))
    print("frame interval ms: median %.2f, mean %.2f" % (np.median(dt) * 1000, dt.mean() * 1000))
    if not np.all(dt > 0):
        print("warning: pts_s is not strictly increasing")

    if not ROOT_VIDEO.exists():
        raise FileNotFoundError(str(ROOT_VIDEO))
    info = ffprobe_json(["-select_streams", "v:0", "-show_streams", str(ROOT_VIDEO)])["streams"][0]
    root_rotation = None
    for sd in info.get("side_data_list", []):
        if "rotation" in sd:
            root_rotation = int(round(float(sd["rotation"])))
    if root_rotation is None and "rotate" in info.get("tags", {}):
        root_rotation = -int(info["tags"]["rotate"])
    if root_rotation is None:
        root_rotation = -90
        print("rotation not found in the file, using -90")
    rot_code = {90: cv2.ROTATE_90_COUNTERCLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_CLOCKWISE}.get(root_rotation % 360)
    print("root stored size: %dx%d, display rotation: %d" % (info["width"], info["height"], root_rotation))

    raw_manifest = pd.read_csv(REPORTS / "raw_manifest.csv", dtype=str)
    row = raw_manifest[raw_manifest["kind"] == "root_video"]
    root_rel = row.iloc[0]["rel_path"]
    root_sha = row.iloc[0]["sha256"]
    print("root video in manifest:", root_rel, root_sha[:12])

    free_gb = shutil.disk_usage(PROJECT_ROOT).free / 1e9
    print("free disk: %.1f GB" % free_gb)
    if free_gb < MIN_FREE_GB:
        raise SystemExit("less than %d GB free, stopping." % MIN_FREE_GB)
except Exception as e:
    print("setup problem:", type(e).__name__, e)

config loaded: ['source_root', 'directories', 'devices', 'nominal_fps', 'chunk_seconds_hypothesis', 'thresholds', 'time', 'identity']
ffmpeg version 9.0.2-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
ffprobe version 9.0.2-full_build-www.gyan.dev Copyright (c) 2007-2026 the FFmpeg developers
root_start_utc: 2026-10-11T14:38:44.654044+00:00
npz keys: ['pts_s', 'motion']
root frames: 79401  duration: 2621.3 s
frame interval ms: median 33.29, mean 33.01
root stored size: 1920x1080, display rotation: -90
root video in manifest: video_20261011_200844.mp4 ae387c43cf89
free disk: 46.6 GB


## 2. Frame ranges

Turns the capture start and stop times of each episode into frame ranges of the root video.

In [2]:
try:
    ep_ids = [str(e) for e in read_json(REPORTS / "06_summary.json")["episode_ids"]]
    print("episodes:", ep_ids)
    try:
        print("lag_s from 08b (info only):", read_json(REPORTS / "08b_exo_sync.json").get("lag_s"))
    except Exception as e:
        print("could not read 08b_exo_sync.json:", e)

    plan_rows = []
    for ep in ep_ids:
        meta = read_json(EPISODES / ep / "episode_metadata_draft.json")
        start_iso, stop_iso = meta["capture_start_iso"], meta["capture_stop_iso"]
        r_start = (iso_to_ns(start_iso) - root_start_ns) / 1e9
        r_stop = (iso_to_ns(stop_iso) - root_start_ns) / 1e9
        idx = np.flatnonzero((pts >= r_start) & (pts < r_stop))
        if len(idx) == 0:
            print(ep, "no frames in range", round(r_start, 2), round(r_stop, 2))
            continue
        first, last = int(idx[0]), int(idx[-1])
        if len(idx) != last - first + 1:
            print(ep, "warning: frame range is not contiguous")
        dt_ms = np.diff(pts[first:last + 1]) * 1000
        margin_before = r_start - pts[0]
        margin_after = pts[-1] - r_stop
        plan_rows.append(dict(
            episode_id=ep, capture_start_iso=start_iso, capture_stop_iso=stop_iso,
            r_start_s=r_start, r_stop_s=r_stop, first_idx=first, last_idx=last, n_frames=len(idx),
            root_t_first_s=float(pts[first]), root_t_last_s=float(pts[last]),
            margin_before_s=float(margin_before), margin_after_s=float(margin_after),
            inside_root=bool(margin_before >= 0 and margin_after >= 0),
            mean_fps=float((len(idx) - 1) / (pts[last] - pts[first])),
            share_lt25ms=float((dt_ms < 25).mean()), share_gt40ms=float((dt_ms > 40).mean()),
        ))
        q = plan_rows[-1]
        print("%s: frames %d-%d (%d), root %.1f-%.1f s, margin %.1f / %.1f s, %.2f fps, <25ms %.1f%%, >40ms %.1f%%" % (
            ep, first, last, q["n_frames"], q["root_t_first_s"], q["root_t_last_s"],
            margin_before, margin_after, q["mean_fps"], 100 * q["share_lt25ms"], 100 * q["share_gt40ms"]))
        if not q["inside_root"]:
            print("  range is not inside the root recording")

    plan = pd.DataFrame(plan_rows)
    P = {r["episode_id"]: r for r in plan_rows}
    plan_path = REPORTS / "08c_frame_plan.csv"
    if not need(plan_path):
        old = pd.read_csv(plan_path, dtype={"episode_id": str})
        if list(old["first_idx"]) != list(plan["first_idx"]) or list(old["n_frames"]) != list(plan["n_frames"]):
            print("note: saved plan differs from this run, set FORCE = True to replace it")
    save_csv(plan, plan_path)
except Exception as e:
    print("step 2 problem:", type(e).__name__, e)

episodes: ['vendorX_2026-10-11_op01_000001', 'vendorX_2026-10-11_op01_000002', 'vendorX_2026-10-11_op01_000003', 'vendorX_2026-10-11_op01_000004']
lag_s from 08b (info only): 105.81666666666666
vendorX_2026-10-11_op01_000001: frames 12005-33273 (21269), root 377.9-1085.8 s, margin 377.8 / 1535.5 s, 30.04 fps, <25ms 0.0%, >40ms 0.0%
vendorX_2026-10-11_op01_000002: frames 34836-44238 (9403), root 1137.9-1450.8 s, margin 1137.8 / 1170.5 s, 30.04 fps, <25ms 0.0%, >40ms 0.0%
vendorX_2026-10-11_op01_000003: frames 46042-56464 (10423), root 1510.9-1857.8 s, margin 1510.8 / 763.5 s, 30.04 fps, <25ms 0.0%, >40ms 0.0%
vendorX_2026-10-11_op01_000004: frames 61062-79085 (18024), root 2010.9-2610.8 s, margin 2010.8 / 10.5 s, 30.04 fps, <25ms 0.0%, >40ms 0.0%
kept existing 08c_frame_plan.csv


## 3. CRF test

Encodes an excerpt of 900 frames from episode 1 with each CRF and compares it with a lossless reference. The CRF is picked from the table.

In [3]:
def vf_chain(first, last):
    return "trim=start_frame=" + str(first) + ":end_frame=" + str(last + 1) + ",setpts=N/(30*TB),format=yuv420p"


keep_args = ["-pix_fmt", "yuv420p", "-g", "30", "-keyint_min", "30", "-sc_threshold", "0", "-bf", "0",
             "-color_range", "tv", "-colorspace", "bt709", "-color_primaries", "bt709", "-color_trc", "bt709",
             "-x264-params", "repeat-headers=1:aud=1"]
lossless_args = ["-c:v", "libx264", "-preset", PRESET, "-qp", "0"] + keep_args


def lossy_args(crf):
    return ["-c:v", "libx264", "-preset", PRESET, "-crf", str(crf), "-profile:v", "high", "-level", "4.2"] + keep_args


def ffmpeg_cmd(first, last, codec_args, out):
    n = last - first + 1
    return (["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostats", "-nostdin", "-y", "-i", str(ROOT_VIDEO),
             "-vf", vf_chain(first, last), "-fps_mode", "passthrough", "-frames:v", str(n), "-an"]
            + codec_args + ["-f", "h264", str(out)])


def measure(test, ref):
    graph = ("[0:v]setpts=N/(30*TB)[a];[1:v]setpts=N/(30*TB)[b];[a]split[a1][a2];[b]split[b1][b2];"
             "[a1][b1]ssim[o1];[a2][b2]psnr[o2]")
    cmd = ["ffmpeg", "-hide_banner", "-nostdin",
           "-framerate", "30", "-f", "h264", "-i", str(test),
           "-framerate", "30", "-f", "h264", "-i", str(ref),
           "-filter_complex", graph,
           "-map", "[o1]", "-f", "null", "-",
           "-map", "[o2]", "-f", "null", "-"]
    r = run(cmd)
    ss = re.search(r"SSIM Y:.*?All:([0-9.]+)", r.stderr)
    ps = re.search(r"PSNR y:.*?average:(\S+)", r.stderr)
    if ss is None or ps is None:
        raise RuntimeError("could not read ssim or psnr: " + r.stderr[-300:])
    return float(ss.group(1)), float(ps.group(1))


try:
    ep1 = ep_ids[0]
    a, b = P[ep1]["first_idx"], P[ep1]["last_idx"]
    n1 = b - a + 1
    n_test = 900
    if n1 < n_test:
        raise ValueError("episode 1 has only %d frames" % n1)
    t_first = a + (n1 - n_test) // 2
    t_last = t_first + n_test - 1
    t_dur = float(pts[min(t_last + 1, len(pts) - 1)] - pts[t_first])
    print("excerpt: frames %d-%d of the root, %.2f s" % (t_first, t_last, t_dur))

    test_path = REPORTS / "08c_encode_test.csv"
    if need(test_path):
        tmp = INTERIM / "08c_tmp"
        tmp.mkdir(parents=True, exist_ok=True)
        try:
            ref = tmp / "ref_lossless.h264"
            r = run(ffmpeg_cmd(t_first, t_last, lossless_args, ref))
            if r.returncode != 0 or not ref.exists():
                raise RuntimeError("reference encode failed: " + r.stderr[-300:])
            rows = []
            for crf in tqdm(CRF_CANDIDATES, desc="crf test"):
                out = tmp / ("test_crf" + str(crf) + ".h264")
                r = run(ffmpeg_cmd(t_first, t_last, lossy_args(crf), out))
                if r.returncode != 0 or not out.exists():
                    raise RuntimeError("test encode failed: " + r.stderr[-300:])
                size = out.stat().st_size
                ssim, psnr = measure(out, ref)
                rows.append(dict(crf=crf, bytes=size, mbps=round(size * 8 / t_dur / 1e6, 3), ssim=ssim, psnr_db=psnr))
            enc_test = pd.DataFrame(rows)
        finally:
            shutil.rmtree(tmp, ignore_errors=True)
        save_csv(enc_test, test_path)
    else:
        enc_test = pd.read_csv(test_path)
        print("kept existing", test_path.name)
    enc_test = enc_test.sort_values("crf").reset_index(drop=True)
    print(enc_test.to_string(index=False))

    if enc_test["psnr_db"].max() < 38:
        print("highest psnr is below 38 dB, the measurement is probably misaligned.")
        raise SystemExit("stopping: psnr below 38 dB for every crf")

    good = enc_test[(enc_test["ssim"] >= TARGET_SSIM) & (enc_test["mbps"] <= BITRATE_CAP_MBPS)]
    chosen_flag = False
    if len(good):
        pick = good.iloc[-1]
        chosen_why = "highest crf with ssim >= %.3f and bitrate <= %.1f Mbps" % (TARGET_SSIM, BITRATE_CAP_MBPS)
    else:
        chosen_flag = True
        reach = enc_test[enc_test["ssim"] >= TARGET_SSIM]
        if len(reach):
            pick = reach.sort_values("mbps").iloc[0]
            chosen_why = "no crf is under the bitrate cap, lowest bitrate that reaches the ssim target"
        else:
            pick = enc_test.sort_values("ssim").iloc[-1]
            chosen_why = "no crf reaches the ssim target, took the best ssim"
    chosen_crf = int(pick["crf"])
    print("chosen crf %d (ssim %.4f, %.2f Mbps): %s" % (chosen_crf, pick["ssim"], pick["mbps"], chosen_why))
    if chosen_flag:
        print("FLAG: check this choice")
except Exception as e:
    print("step 3 problem:", type(e).__name__, e)

excerpt: frames 22189-23088 of the root, 29.96 s
kept existing 08c_encode_test.csv
 crf    bytes  mbps     ssim   psnr_db
  24 11019223 2.942 0.986596 42.435430
  26  7360949 1.966 0.983983 41.229205
  28  5151628 1.376 0.980913 40.076968
  30  3794502 1.013 0.977216 38.969127
  32  2910851 0.777 0.972598 37.878635
chosen crf 28 (ssim 0.9809, 1.38 Mbps): highest crf with ssim >= 0.980 and bitrate <= 8.0 Mbps


## 4. Final encode

One ffmpeg call per episode, cutting by frame number from the root video.

In [4]:
try:
    final_jobs = []
    for ep in ep_ids:
        p = P[ep]
        out = EPISODES / ep / "video" / "cam_ext.h264"
        part = out.with_name(out.name + ".part")
        cmd = ffmpeg_cmd(p["first_idx"], p["last_idx"], lossy_args(chosen_crf), out)
        run_cmd = ffmpeg_cmd(p["first_idx"], p["last_idx"], lossy_args(chosen_crf), part)
        final_jobs.append(dict(ep=ep, n=p["n_frames"], out=out, part=part, cmd=cmd, run_cmd=run_cmd))

    print("first job:")
    print(subprocess.list2cmdline(final_jobs[0]["cmd"]))

    enc_info = {}
    t_all = time.time()
    for job in tqdm(final_jobs, desc="encode"):
        ep, out = job["ep"], job["out"]
        try:
            out.parent.mkdir(parents=True, exist_ok=True)
            if not need(out):
                tqdm.write(ep + ": exists, skipped")
                enc_info[ep] = dict(skipped=True, seconds=0.0)
                continue
            t0 = time.time()
            # encode to a .part file and rename when done, so a stopped run leaves no half file
            try:
                r = run(job["run_cmd"])
            except BaseException:
                job["part"].unlink(missing_ok=True)
                raise
            sec = time.time() - t0
            if r.returncode != 0 or not job["part"].exists():
                job["part"].unlink(missing_ok=True)
                tqdm.write(ep + ": encode failed: " + r.stderr[-300:])
                continue
            job["part"].replace(out)
            wh = ffprobe_json(["-f", "h264", "-select_streams", "v:0", "-show_entries", "stream=width,height", str(out)])["streams"][0]
            enc_info[ep] = dict(skipped=False, seconds=sec, width=wh["width"], height=wh["height"])
            msg = "%s: %d frames, %dx%d, %.0f s, %.1f fps" % (ep, job["n"], wh["width"], wh["height"], sec, job["n"] / sec)
            if (wh["width"], wh["height"]) != (1080, 1920):
                msg += "  <- not 1080x1920"
            tqdm.write(msg)
        except Exception as e:
            tqdm.write(ep + " problem: " + type(e).__name__ + " " + str(e))

    done = [j for j in final_jobs if j["ep"] in enc_info and not enc_info[j["ep"]]["skipped"]]
    secs = sum(enc_info[j["ep"]]["seconds"] for j in done)
    frames_done = sum(j["n"] for j in done)
    print("elapsed %.0f s in total" % (time.time() - t_all))
    if secs > 0:
        print("encoded %d frames at %.1f fps" % (frames_done, frames_done / secs))

    cmd_path = REPORTS / "08c_commands.txt"
    if need(cmd_path):
        with open(cmd_path, "w", encoding="utf-8") as f:
            for j in final_jobs:
                f.write("# " + j["ep"] + "\n" + subprocess.list2cmdline(j["cmd"]) + "\n\n")
        print("saved", cmd_path.name)
    else:
        print("kept existing", cmd_path.name)
except Exception as e:
    print("step 4 problem:", type(e).__name__, e)

first job:
ffmpeg -hide_banner -loglevel error -nostats -nostdin -y -i "c:\workstation\3RD YEAR\ego_pipeline\raw\video_20261011_200844.mp4" -vf trim=start_frame=12005:end_frame=33274,setpts=N/(30*TB),format=yuv420p -fps_mode passthrough -frames:v 21269 -an -c:v libx264 -preset veryfast -crf 28 -profile:v high -level 4.2 -pix_fmt yuv420p -g 30 -keyint_min 30 -sc_threshold 0 -bf 0 -color_range tv -colorspace bt709 -color_primaries bt709 -color_trc bt709 -x264-params repeat-headers=1:aud=1 -f h264 "c:\workstation\3RD YEAR\ego_pipeline\episodes\vendorX_2026-10-11_op01_000001\video\cam_ext.h264"


encode: 100%|██████████| 4/4 [00:00<00:00, 95.99it/s]

vendorX_2026-10-11_op01_000001: exists, skipped
vendorX_2026-10-11_op01_000002: exists, skipped
vendorX_2026-10-11_op01_000003: exists, skipped
vendorX_2026-10-11_op01_000004: exists, skipped
elapsed 0 s in total
kept existing 08c_commands.txt


## 5. Access units and timestamps

Splits each stream into access units and builds the timestamp file from the root pts values.

In [5]:
def slice_type_of(chunk):
    # reads first_mb_in_slice and slice_type (both exp-golomb) from the start of a slice nal
    data = bytes(chunk).replace(b"\x00\x00\x03", b"\x00\x00")
    bits = int.from_bytes(data, "big")
    total = len(data) * 8
    pos = 0
    vals = []
    for _ in range(2):
        zeros = 0
        while True:
            if pos + zeros >= total:
                return -1
            if (bits >> (total - 1 - pos - zeros)) & 1:
                break
            zeros += 1
        pos += zeros + 1
        if pos + zeros > total:
            return -1
        suffix = (bits >> (total - pos - zeros)) & ((1 << zeros) - 1) if zeros else 0
        vals.append((1 << zeros) - 1 + suffix)
        pos += zeros
    return vals[1] % 5


def split_aus(path):
    nal_pos, nal_type, lead, stype = [], [], [], []
    with open(path, "rb") as f, mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
        total = len(mm)
        p = mm.find(b"\x00\x00\x01")
        while p != -1:
            t = mm[p + 3] & 31
            nal_pos.append(p)
            nal_type.append(t)
            lead.append(1 if p > 0 and mm[p - 1] == 0 else 0)
            stype.append(slice_type_of(mm[p + 4:p + 16]) if t in (1, 5) else -1)
            p = mm.find(b"\x00\x00\x01", p + 3)
    types = np.array(nal_type, dtype=np.int16)
    stype = np.array(stype, dtype=np.int16)
    starts = np.array(nal_pos, dtype=np.int64) - np.array(lead, dtype=np.int64)
    is_aud = types == 9
    first_nal = np.flatnonzero(is_aud)
    n_au = len(first_nal)
    au_id = np.cumsum(is_aud) - 1
    offsets = starts[first_nal]
    sizes = np.diff(np.append(offsets, total))

    def per_au(mask):
        return np.bincount(au_id[mask & (au_id >= 0)], minlength=n_au) > 0

    return dict(
        n_au=n_au, offsets=offsets, sizes=sizes,
        is_key=per_au(types == 5), has_sps=per_au(types == 7), has_pps=per_au(types == 8),
        has_b=per_au(stype == 1), has_slice=per_au((types == 1) | (types == 5)),
        junk_before_first=int(offsets[0]) if n_au else -1,
    )


try:
    au_info, ts_info = {}, {}
    shown_keys = False
    for ep in ep_ids:
        try:
            video_dir = EPISODES / ep / "video"
            h264 = video_dir / "cam_ext.h264"
            if not h264.exists():
                print(ep, "no cam_ext.h264, skipped")
                continue
            p = P[ep]
            n = p["n_frames"]

            au = split_aus(h264)
            au_info[ep] = au
            n_au = au["n_au"]
            print("%s: %d units for %d frames, %d keyframes, %d with b slices" % (
                ep, n_au, n, au["is_key"].sum(), au["has_b"].sum()))
            if n_au != n:
                print("  unit count does not match the frame count")
            if au["junk_before_first"] != 0:
                print("  data before the first unit:", au["junk_before_first"], "bytes")

            au_path = video_dir / "cam_ext_au.npz"
            if need(au_path):
                np.savez(au_path, offsets=au["offsets"], sizes=au["sizes"],
                         is_key=au["is_key"].astype(np.uint8), frame_idx=np.arange(n_au, dtype=np.int32))

            if not shown_keys:
                others = sorted(video_dir.glob("cam_*_au.npz"))
                others = [o for o in others if o.name != "cam_ext_au.npz"]
                if others:
                    print("  keys in", others[0].name + ":", np.load(others[0]).files, "| cam_ext_au.npz:", ["offsets", "sizes", "is_key", "frame_idx"])
                shown_keys = True

            ids = np.arange(p["first_idx"], p["first_idx"] + n)
            pts_ep = pts[ids]
            ts_ns = root_start_ns + np.rint(pts_ep * 1e9).astype(np.int64)
            prev_ms = np.round(np.diff(pts, prepend=np.nan)[ids] * 1000, 3)
            flag = np.zeros(n, dtype=np.uint8)
            flag[prev_ms >= 50] = 1
            flag[prev_ms < 20] = 2
            increasing = bool(np.all(np.diff(ts_ns) > 0))
            ts_info[ep] = dict(increasing=increasing, n_gap=int((flag == 1).sum()), n_short=int((flag == 2).sum()))
            print("  timestamps strictly increasing:", increasing, "| flag 1: %d, flag 2: %d" % (
                ts_info[ep]["n_gap"], ts_info[ep]["n_short"]))

            ts_path = video_dir / "cam_ext_timestamps.npz"
            if need(ts_path):
                np.savez(ts_path, pts_s=pts_ep, ts_utc_ns=ts_ns, flag=flag, root_frame_idx=ids.astype(np.int32))
        except Exception as e:
            print(ep, "problem:", type(e).__name__, e)
except Exception as e:
    print("step 5 problem:", type(e).__name__, e)

vendorX_2026-10-11_op01_000001: 21269 units for 21269 frames, 709 keyframes, 0 with b slices
  keys in cam_high_au.npz: ['offsets', 'sizes', 'is_key', 'n_sps', 'n_pps', 'n_vcl', 'frame_index', 'first_is_aud'] | cam_ext_au.npz: ['offsets', 'sizes', 'is_key', 'frame_idx']
  timestamps strictly increasing: True | flag 1: 0, flag 2: 0
vendorX_2026-10-11_op01_000002: 9403 units for 9403 frames, 314 keyframes, 0 with b slices
  timestamps strictly increasing: True | flag 1: 0, flag 2: 0
vendorX_2026-10-11_op01_000003: 10423 units for 10423 frames, 348 keyframes, 0 with b slices
  timestamps strictly increasing: True | flag 1: 0, flag 2: 0
vendorX_2026-10-11_op01_000004: 18024 units for 18024 frames, 601 keyframes, 0 with b slices
  timestamps strictly increasing: True | flag 1: 0, flag 2: 0


## 6. Standard checks

Checks each stream with ffprobe and decodes every frame with PyAV to look for repeated and black frames.

In [6]:
def probe_stream(path):
    entries = "stream=codec_name,profile,level,width,height,pix_fmt,color_range,color_space,color_primaries,color_transfer,nb_read_frames"
    j = ffprobe_json(["-f", "h264", "-count_frames", "-select_streams", "v:0", "-show_entries", entries, str(path)])
    return j["streams"][0]


def first_pict_types(path, count=300):
    r = run(["ffprobe", "-v", "error", "-f", "h264", "-select_streams", "v:0", "-read_intervals", "%+#" + str(count),
             "-show_entries", "frame=pict_type", "-of", "csv=p=0", str(path)])
    types = [ln.strip().strip(",") for ln in r.stdout.splitlines() if ln.strip()]
    return types[:count]


def thumb200(rgb):
    h = int(round(rgb.shape[0] * 200 / rgb.shape[1]))
    return cv2.resize(rgb, (200, h), interpolation=cv2.INTER_AREA)


def decode_checks(path, n_expect):
    cont = av.open(str(path), format="h264")
    stream = cont.streams.video[0]
    stream.thread_type = "AUTO"
    crcs, dark = [], []
    first_rgb, last_frame, count = None, None, 0
    for frame in tqdm(cont.decode(stream), total=n_expect, desc=path.parent.parent.name, mininterval=2):
        plane = frame.planes[0]
        raw = np.frombuffer(plane, dtype=np.uint8)
        rows = raw.size // plane.line_size
        luma = raw[:rows * plane.line_size].reshape(rows, plane.line_size)[:frame.height, :frame.width]
        crcs.append(zlib.crc32(np.ascontiguousarray(luma)))
        sub = luma[::4, ::4]
        dark.append(bool(sub.mean() < 16 and sub.std() < 8))
        if count == 0:
            first_rgb = frame.to_ndarray(format="rgb24")
        last_frame = frame
        count += 1
    last_rgb = last_frame.to_ndarray(format="rgb24") if last_frame is not None else None
    cont.close()
    same = [crcs[i] == crcs[i - 1] for i in range(1, len(crcs))]
    same_runs = [x + 1 for x in run_lengths(same)]
    dark_runs = run_lengths(dark)
    res = dict(decoded_frames=count, identical_runs=len(same_runs), identical_longest=max(same_runs, default=0),
               black_frames=int(sum(dark)), black_runs=len(dark_runs), black_longest=max(dark_runs, default=0))
    return res, first_rgb, last_rgb


try:
    std_path = REPORTS / "08c_standard_checks.csv"
    thumbs = {}
    if need(std_path):
        rows = []
        for ep in ep_ids:
            try:
                h264 = EPISODES / ep / "video" / "cam_ext.h264"
                pr = probe_stream(h264)
                types = Counter(first_pict_types(h264))
                dec, first_rgb, last_rgb = decode_checks(h264, P[ep]["n_frames"])
                thumbs[ep] = (thumb200(first_rgb), thumb200(last_rgb), 0, dec["decoded_frames"] - 1)
                rows.append(dict(
                    episode_id=ep, codec=pr.get("codec_name"), profile=pr.get("profile"), level=pr.get("level"),
                    width=pr.get("width"), height=pr.get("height"), pix_fmt=pr.get("pix_fmt"),
                    color_range=pr.get("color_range"), color_space=pr.get("color_space"),
                    plan_frames=P[ep]["n_frames"], probe_frames=int(pr["nb_read_frames"]),
                    decoded_frames=dec["decoded_frames"],
                    frames_match_plan=bool(int(pr["nb_read_frames"]) == P[ep]["n_frames"] == dec["decoded_frames"]),
                    first300_I=types.get("I", 0), first300_P=types.get("P", 0), first300_B=types.get("B", 0),
                    identical_runs=dec["identical_runs"], identical_longest=dec["identical_longest"],
                    black_frames=dec["black_frames"], black_runs=dec["black_runs"], black_longest=dec["black_longest"],
                ))
            except Exception as e:
                print(ep, "problem:", type(e).__name__, e)
        std_df = pd.DataFrame(rows)
        save_csv(std_df, std_path)
    else:
        std_df = pd.read_csv(std_path, dtype={"episode_id": str})
        print("kept existing", std_path.name)

    S = {r["episode_id"]: r for r in std_df.to_dict("records")}
    for ep, s in S.items():
        print("%s: %s %s level %s, %sx%s %s %s %s | frames plan %d probe %d decoded %d | first 300: I %d P %d B %d" % (
            ep, s["codec"], s["profile"], s["level"], s["width"], s["height"], s["pix_fmt"], s["color_range"], s["color_space"],
            s["plan_frames"], s["probe_frames"], s["decoded_frames"], s["first300_I"], s["first300_P"], s["first300_B"]))
        print("   identical runs: %d (longest %d) | black frames: %d in %d runs (longest %d)" % (
            s["identical_runs"], s["identical_longest"], s["black_frames"], s["black_runs"], s["black_longest"]))
except Exception as e:
    print("step 6 problem:", type(e).__name__, e)

kept existing 08c_standard_checks.csv
vendorX_2026-10-11_op01_000001: h264 High level 42, 1080x1920 yuv420p tv bt709 | frames plan 21269 probe 21269 decoded 21269 | first 300: I 10 P 290 B 0
   identical runs: 0 (longest 0) | black frames: 0 in 0 runs (longest 0)
vendorX_2026-10-11_op01_000002: h264 High level 42, 1080x1920 yuv420p tv bt709 | frames plan 9403 probe 9403 decoded 9403 | first 300: I 10 P 290 B 0
   identical runs: 0 (longest 0) | black frames: 0 in 0 runs (longest 0)
vendorX_2026-10-11_op01_000003: h264 High level 42, 1080x1920 yuv420p tv bt709 | frames plan 10423 probe 10423 decoded 10423 | first 300: I 10 P 290 B 0
   identical runs: 0 (longest 0) | black frames: 0 in 0 runs (longest 0)
vendorX_2026-10-11_op01_000004: h264 High level 42, 1080x1920 yuv420p tv bt709 | frames plan 18024 probe 18024 decoded 18024 | first 300: I 10 P 290 B 0
   identical runs: 0 (longest 0) | black frames: 0 in 0 runs (longest 0)


## 7. Frame accuracy

Compares random output frames with the matching root frames at offsets from -2 to 2. The smallest difference should be at offset 0.

In [7]:
SEED = 0
N_SAMPLES = 20
OFFSETS = [-2, -1, 0, 1, 2]


def small_gray(g):
    # 64 x 36 as asked
    return cv2.resize(g, (64, 36), interpolation=cv2.INTER_AREA).astype(np.float32)


def grab_frames(path, wanted, fmt, desc):
    # decode in order and keep a small grey copy of the wanted frame numbers
    got = {}
    last = max(wanted)
    cont = av.open(str(path), format=fmt)
    stream = cont.streams.video[0]
    stream.thread_type = "AUTO"
    with tqdm(total=last + 1, desc=desc, mininterval=2) as bar:
        for i, frame in enumerate(cont.decode(stream)):
            if i in wanted:
                g = frame.to_ndarray(format="gray")
                if rot_code is not None and g.shape[0] < g.shape[1]:
                    g = cv2.rotate(g, rot_code)
                got[i] = small_gray(g)
            bar.update(1)
            if i >= last:
                break
    cont.close()
    return got


try:
    align_path = REPORTS / "08c_alignment_check.csv"
    if need(align_path):
        rng = np.random.default_rng(SEED)
        picks, wanted_src = {}, set()
        for ep in ep_ids:
            n = P[ep]["n_frames"]
            ks = np.sort(rng.choice(np.arange(2, n - 2), size=min(N_SAMPLES, n - 4), replace=False))
            picks[ep] = [int(k) for k in ks]
            for k in picks[ep]:
                for o in OFFSETS:
                    wanted_src.add(P[ep]["first_idx"] + k + o)
        src = grab_frames(ROOT_VIDEO, wanted_src, None, "root video")
        rows = []
        for ep in ep_ids:
            try:
                got = grab_frames(EPISODES / ep / "video" / "cam_ext.h264", set(picks[ep]), "h264", ep)
                first = P[ep]["first_idx"]
                for k in picks[ep]:
                    mads = [float(np.abs(got[k] - src[first + k + o]).mean()) for o in OFFSETS]
                    rows.append(dict(episode_id=ep, out_frame=k, root_frame=first + k,
                                     mad_m2=mads[0], mad_m1=mads[1], mad_0=mads[2], mad_p1=mads[3], mad_p2=mads[4],
                                     best_offset=OFFSETS[int(np.argmin(mads))]))
            except Exception as e:
                print(ep, "problem:", type(e).__name__, e)
        A = pd.DataFrame(rows)
        save_csv(A, align_path)
    else:
        A = pd.read_csv(align_path, dtype={"episode_id": str})
        print("kept existing", align_path.name)

    for ep in ep_ids:
        sub = A[A["episode_id"] == ep]
        if len(sub):
            print("%s: %d samples, minimum at offset 0 in %.0f%%" % (ep, len(sub), 100 * (sub["best_offset"] == 0).mean()))
    print("all: %.1f%% at offset 0" % (100 * (A["best_offset"] == 0).mean()))
except Exception as e:
    print("step 7 problem:", type(e).__name__, e)

kept existing 08c_alignment_check.csv
vendorX_2026-10-11_op01_000001: 20 samples, minimum at offset 0 in 70%
vendorX_2026-10-11_op01_000002: 20 samples, minimum at offset 0 in 75%
vendorX_2026-10-11_op01_000003: 20 samples, minimum at offset 0 in 80%
vendorX_2026-10-11_op01_000004: 20 samples, minimum at offset 0 in 85%
all: 77.5% at offset 0


In [8]:
# sequence alignment
WIN = 1000
STEP = 500
LAGS = list(range(-5, 6))
MIN_PEAK_CORR = 0.3


def decode_motion(path, n_expect):
    # mean absolute difference to the previous frame on the 36 x 64 grey downscale, frame 0 has no previous frame
    cont = av.open(str(path), format="h264")
    stream = cont.streams.video[0]
    stream.thread_type = "AUTO"
    prev, vals = None, []
    for frame in tqdm(cont.decode(stream), total=n_expect, desc=path.parent.parent.name, mininterval=2):
        cur = small_gray(frame.to_ndarray(format="gray"))
        vals.append(np.nan if prev is None else float(np.abs(cur - prev).mean()))
        prev = cur
    cont.close()
    return np.array(vals, dtype=np.float64)


def lag_corr(out_m, src_m, first, start, lag):
    k = np.arange(start, start + WIN)
    j = first + k + lag
    inside = (j >= 0) & (j < len(src_m))
    a, b = out_m[k[inside]], src_m[j[inside]]
    good = np.isfinite(a) & np.isfinite(b)
    a, b = a[good], b[good]
    if len(a) < 3 or a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


try:
    seq_path = REPORTS / "08c_sequence_alignment.csv"
    if need(seq_path):
        src_motion = np.asarray(np.load(INTERIM / "root_motion_30hz.npz")["motion"], dtype=np.float64)
        print("src_motion frames:", len(src_motion))
        seq_rows = []
        for ep in ep_ids:
            try:
                out_motion = decode_motion(EPISODES / ep / "video" / "cam_ext.h264", P[ep]["n_frames"])
                if len(out_motion) != P[ep]["n_frames"]:
                    print(ep, "decoded", len(out_motion), "frames, plan", P[ep]["n_frames"])
                first = P[ep]["first_idx"]
                for start in range(0, len(out_motion) - WIN + 1, STEP):
                    cs = [lag_corr(out_motion, src_motion, first, start, lag) for lag in LAGS]
                    if np.all(np.isnan(cs)):
                        print(ep, "window", start, "has no usable correlation, skipped")
                        continue
                    i_best = int(np.nanargmax(cs))
                    seq_rows.append(dict(episode_id=ep, window_start=start, best_lag=LAGS[i_best],
                                         peak_corr=cs[i_best], corr_at_0=cs[LAGS.index(0)]))
            except Exception as e:
                print(ep, "problem:", type(e).__name__, e)
        SQ = pd.DataFrame(seq_rows, columns=["episode_id", "window_start", "best_lag", "peak_corr", "corr_at_0"])
        save_csv(SQ, seq_path)
    else:
        SQ = pd.read_csv(seq_path, dtype={"episode_id": str})
        print("kept existing", seq_path.name)

    for ep in ep_ids:
        sub = SQ[SQ["episode_id"] == ep]
        if len(sub) == 0:
            print("%s: no windows" % ep)
            continue
        print("%s: %d windows, best lag 0 in %.0f%%, largest |best lag| %d, lowest peak corr %.3f" % (
            ep, len(sub), 100 * (sub["best_lag"] == 0).mean(), sub["best_lag"].abs().max(), sub["peak_corr"].min()))
        for r in sub[sub["best_lag"] != 0].itertuples():
            print("   window %d: best lag %+d, peak corr %.3f, corr at 0 %.3f" % (r.window_start, r.best_lag, r.peak_corr, r.corr_at_0))

    # test x6b, added to the saved reports because the hypothesis cell stays as it is
    strong = SQ[SQ["peak_corr"] >= MIN_PEAK_CORR]
    off = strong[strong["best_lag"] != 0]
    if len(strong) == 0:
        x6b_measured, x6b_result = "no window reaches a peak correlation of 0.3 (%d windows)" % len(SQ), "FLAG"
    else:
        per_ep = "; ".join("%s %d of %d" % (ep, (off["episode_id"] == ep).sum(), (strong["episode_id"] == ep).sum()) for ep in ep_ids)
        x6b_measured = "%d of %d windows with peak corr >= 0.3 have best lag != 0 (%s)" % (len(off), len(strong), per_ep)
        x6b_result = "PASS" if len(off) == 0 else "FLAG"
    print("X6b %s: %s" % (x6b_result, x6b_measured))

    hyp_path = REPORTS / "08c_hypothesis_check.csv"
    hyp_old = pd.read_csv(hyp_path, dtype=str, keep_default_na=False)
    hyp_old = hyp_old[hyp_old["id"] != "X6b"].reset_index(drop=True)
    x6_pos = hyp_old.index[hyp_old["id"] == "X6"]
    cut = int(x6_pos[0]) + 1 if len(x6_pos) else len(hyp_old)
    x6b_row = pd.DataFrame([dict(id="X6b", hypothesis="all windows with a peak correlation of at least 0.3 have best lag 0 in the motion sequence alignment",
                                 measured=x6b_measured, result=x6b_result)])
    hyp2 = pd.concat([hyp_old.iloc[:cut], x6b_row, hyp_old.iloc[cut:]], ignore_index=True)
    hyp2.to_csv(hyp_path, index=False)
    print("saved", hyp_path.name)

    sum_path = REPORTS / "08c_summary.json"
    with open(sum_path, "r", encoding="utf-8") as f:
        summary2 = json.load(f)
    summary2["hypotheses"] = hyp2.to_dict("records")
    summary2["counts"] = {k: int(v) for k, v in hyp2["result"].value_counts().items()}
    with open(sum_path, "w", encoding="utf-8") as f:
        json.dump(summary2, f, indent=2, default=lambda o: o.item() if isinstance(o, np.generic) else str(o))
    print("saved", sum_path.name)

    zip_path = REPORTS / "share_08c.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(REPORTS.glob("08c_*")):
            if f.is_file():
                z.write(f, arcname=f.name)
    print("saved", zip_path.name)
except Exception as e:
    print("step 7b problem:", type(e).__name__, e)

src_motion frames: 79401


vendorX_2026-10-11_op01_000004: 100%|██████████| 18024/18024 [02:35<00:00, 115.54it/s]


saved 08c_sequence_alignment.csv
vendorX_2026-10-11_op01_000001: 41 windows, best lag 0 in 0%, largest |best lag| 1, lowest peak corr 0.527
   window 0: best lag -1, peak corr 0.997, corr at 0 0.901
   window 500: best lag -1, peak corr 0.984, corr at 0 0.894
   window 1000: best lag -1, peak corr 0.899, corr at 0 0.837
   window 1500: best lag -1, peak corr 0.894, corr at 0 0.762
   window 2000: best lag -1, peak corr 0.910, corr at 0 0.716
   window 2500: best lag -1, peak corr 0.909, corr at 0 0.770
   window 3000: best lag -1, peak corr 0.923, corr at 0 0.850
   window 3500: best lag -1, peak corr 0.983, corr at 0 0.968
   window 4000: best lag -1, peak corr 0.983, corr at 0 0.957
   window 4500: best lag -1, peak corr 0.947, corr at 0 0.869
   window 5000: best lag -1, peak corr 0.891, corr at 0 0.761
   window 5500: best lag -1, peak corr 0.883, corr at 0 0.697
   window 6000: best lag -1, peak corr 0.883, corr at 0 0.758
   window 6500: best lag -1, peak corr 0.878, corr at 0 0.

## 8. Size and manifest

Sizes, bitrates and hashes of the outputs, and one manifest file per episode.

In [9]:
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


try:
    run_time_utc = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    man_rows = []
    for ep in ep_ids:
        try:
            p = P[ep]
            video_dir = EPISODES / ep / "video"
            h264 = video_dir / "cam_ext.h264"
            size = h264.stat().st_size
            dur = real_duration(p["first_idx"], p["last_idx"])
            mbps = size * 8 / dur / 1e6
            gb_hour = size / 1e9 / (dur / 3600)
            sha = sha256_of(h264)
            extra = sum((video_dir / name).stat().st_size for name in ["cam_ext_au.npz", "cam_ext_timestamps.npz"])
            s = S.get(ep, {})
            width, height = int(s.get("width", 1080)), int(s.get("height", 1920))
            man_rows.append(dict(episode_id=ep, file="cam_ext.h264", frames=p["n_frames"], width=width, height=height,
                                 duration_s=round(dur, 3), bytes=size, mbps=round(mbps, 3), gb_per_hour=round(gb_hour, 3),
                                 sha256=sha, combined_bytes=size + extra))
            print("%s: %.2f GB, %.2f Mbps, %.2f GB per hour, combined %.2f GB" % (ep, size / 1e9, mbps, gb_hour, (size + extra) / 1e9))

            manifest = dict(
                episode_id=ep, camera="cam_ext",
                files=dict(stream="cam_ext.h264", access_units="cam_ext_au.npz", timestamps="cam_ext_timestamps.npz"),
                frames=p["n_frames"], resolution=str(width) + "x" + str(height), width=width, height=height,
                nominal_fps="30/1",
                timing_note="timing inside the stream is normalised to 30 fps, the true timing is in cam_ext_timestamps.npz",
                mean_real_fps=p["mean_fps"], duration_real_s=dur,
                codec=dict(encoder="libx264", preset=PRESET, crf=chosen_crf, profile="high", level="4.2", pix_fmt="yuv420p",
                           gop=30, keyint_min=30, scenecut=False, b_frames=0, repeat_headers=True,
                           access_unit_delimiters=True, color_range="tv", colorspace="bt709",
                           color_primaries="bt709", color_trc="bt709", container="raw annex b"),
                display_rotation_applied=True, display_rotation_deg=root_rotation, audio_removed=True,
                source=dict(file=root_rel, sha256=root_sha, first_frame=p["first_idx"], last_frame=p["last_idx"],
                            root_time_first_s=p["root_t_first_s"], root_time_last_s=p["root_t_last_s"],
                            capture_start_iso=p["capture_start_iso"], capture_stop_iso=p["capture_stop_iso"]),
                bytes=size, mbps=mbps, sha256=sha, ffmpeg_version=ffmpeg_line, run_time_utc=run_time_utc,
            )
            man_path = video_dir / "video_manifest_exo.json"
            if need(man_path):
                with open(man_path, "w", encoding="utf-8") as f:
                    json.dump(manifest, f, indent=2)
        except Exception as e:
            print(ep, "problem:", type(e).__name__, e)

    manifest_df = pd.DataFrame(man_rows)
    save_csv(manifest_df, REPORTS / "08c_video_manifest.csv")
    print("total: %.2f GB" % (manifest_df["bytes"].sum() / 1e9))
except Exception as e:
    print("step 8 problem:", type(e).__name__, e)

vendorX_2026-10-11_op01_000001: 0.14 GB, 1.60 Mbps, 0.72 GB per hour, combined 0.14 GB
vendorX_2026-10-11_op01_000002: 0.06 GB, 1.52 Mbps, 0.69 GB per hour, combined 0.06 GB
vendorX_2026-10-11_op01_000003: 0.07 GB, 1.57 Mbps, 0.71 GB per hour, combined 0.07 GB
vendorX_2026-10-11_op01_000004: 0.12 GB, 1.57 Mbps, 0.71 GB per hour, combined 0.12 GB
kept existing 08c_video_manifest.csv
total: 0.39 GB


## 9. Contact sheet

First and last frame of each output.

In [10]:
def read_ends(path):
    cont = av.open(str(path), format="h264")
    stream = cont.streams.video[0]
    stream.thread_type = "AUTO"
    first_rgb, last_frame, count = None, None, 0
    for frame in cont.decode(stream):
        if count == 0:
            first_rgb = frame.to_ndarray(format="rgb24")
        last_frame = frame
        count += 1
    last_rgb = last_frame.to_ndarray(format="rgb24")
    cont.close()
    return thumb200(first_rgb), thumb200(last_rgb), 0, count - 1


try:
    sheet_path = REPORTS / "08c_output_contact.png"
    if need(sheet_path):
        for ep in ep_ids:
            if ep not in thumbs:
                thumbs[ep] = read_ends(EPISODES / ep / "video" / "cam_ext.h264")
        img_h = max(t[0].shape[0] for t in thumbs.values())
        label_h = 30
        canvas = np.full((2 * (img_h + label_h), 200 * len(ep_ids), 3), 255, dtype=np.uint8)
        for c, ep in enumerate(ep_ids):
            first_img, last_img, k_first, k_last = thumbs[ep]
            for r, (img, k) in enumerate([(first_img, k_first), (last_img, k_last)]):
                y0 = r * (img_h + label_h)
                canvas[y0 + label_h:y0 + label_h + img.shape[0], c * 200:(c + 1) * 200] = img
                root_idx = P[ep]["first_idx"] + k
                cv2.putText(canvas, ep[:30], (c * 200 + 3, y0 + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 0), 1, cv2.LINE_AA)
                cv2.putText(canvas, "frame %d (root %d)" % (k, root_idx), (c * 200 + 3, y0 + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 0), 1, cv2.LINE_AA)
        plt.imsave(str(sheet_path), canvas)
        print("saved", sheet_path.name)
        plt.figure(figsize=(2.2 * len(ep_ids), 8))
        plt.imshow(canvas)
        plt.axis("off")
        plt.show()
    else:
        print("kept existing", sheet_path.name)
except Exception as e:
    print("step 9 problem:", type(e).__name__, e)

kept existing 08c_output_contact.png


## 10. Hypothesis tests

Ten checks on the outputs, computed from the results above.

In [11]:
try:
    rows = []

    def add(hid, text, measured, result):
        rows.append(dict(id=hid, hypothesis=text, measured=measured, result=result))

    # x1 frame counts
    parts, ok = [], True
    for ep in ep_ids:
        n, s, units = P[ep]["n_frames"], S[ep], au_info[ep]["n_au"]
        parts.append("%s plan %d probe %d decoded %d units %d" % (ep, n, s["probe_frames"], s["decoded_frames"], units))
        ok = ok and (n == s["probe_frames"] == s["decoded_frames"] == units)
    add("X1", "output frame counts equal the plan", "; ".join(parts), "PASS" if ok else "FLAG")

    # x2 resolution
    sizes = sorted({(int(S[ep]["width"]), int(S[ep]["height"])) for ep in ep_ids})
    add("X2", "one resolution 1080x1920 for all four", ", ".join("%dx%d" % wh for wh in sizes),
        "PASS" if sizes == [(1080, 1920)] else "FLAG")

    # x3 gop structure
    parts, ok = [], True
    for ep in ep_ids:
        au = au_info[ep]
        n_au = au["n_au"]
        key_ok = bool(np.array_equal(au["is_key"], (np.arange(n_au) % 30) == 0))
        n_b = int(au["has_b"].sum()) + int(S[ep]["first300_B"])
        hdr_ok = bool(au["has_sps"][au["is_key"]].all() and au["has_pps"][au["is_key"]].all())
        eq_ok = n_au == P[ep]["n_frames"] and bool(au["has_slice"].all())
        parts.append("%s keys every 30: %s, b: %d, sps/pps: %s, units=frames: %s" % (ep, key_ok, n_b, hdr_ok, eq_ok))
        ok = ok and key_ok and n_b == 0 and hdr_ok and eq_ok
    add("X3", "keyframe every 30 frames, no b-frames, units equal frames, sps and pps at every keyframe", "; ".join(parts), "PASS" if ok else "FLAG")

    # x4 identical frames
    runs = {ep: (int(S[ep]["identical_runs"]), int(S[ep]["identical_longest"])) for ep in ep_ids}
    total_runs = sum(v[0] for v in runs.values())
    add("X4", "no run of identical frames of 2 or more",
        "; ".join("%s runs %d longest %d" % (ep, v[0], v[1]) for ep, v in runs.items()), "PASS" if total_runs == 0 else "FLAG")

    # x5 black frames
    add("X5", "black frame runs (info)",
        "; ".join("%s frames %d runs %d longest %d" % (ep, S[ep]["black_frames"], S[ep]["black_runs"], S[ep]["black_longest"]) for ep in ep_ids), "INFO")

    # x6 alignment
    share = float((A["best_offset"] == 0).mean())
    per_ep = "; ".join("%s %.0f%%" % (ep, 100 * (A[A["episode_id"] == ep]["best_offset"] == 0).mean()) for ep in ep_ids)
    add("X6", "at least 95 percent of alignment samples have the minimum at offset 0",
        "all %.1f%% (%s)" % (100 * share, per_ep), "PASS" if share >= 0.95 else "FLAG")

    # x7 timestamps and range
    inc = all(ts_info[ep]["increasing"] for ep in ep_ids)
    inside = all(P[ep]["inside_root"] for ep in ep_ids)
    add("X7", "timestamps strictly increasing and every range inside the root recording",
        "increasing: %s, inside root: %s" % (inc, inside), "PASS" if inc and inside else "FLAG")

    # x8 frame rate
    add("X8", "mean real fps and irregular intervals (variable frame rate, info)",
        "; ".join("%s %.2f fps, <25ms %.1f%%, >40ms %.1f%%" % (ep, P[ep]["mean_fps"], 100 * P[ep]["share_lt25ms"], 100 * P[ep]["share_gt40ms"]) for ep in ep_ids), "INFO")

    # x9 bitrate
    m = manifest_df.set_index("episode_id")
    add("X9", "bitrate per episode against 5 to 8 Mbps (info)",
        "; ".join("%s %.2f Mbps (%s)" % (ep, m.loc[ep, "mbps"], "in range" if 5 <= m.loc[ep, "mbps"] <= 8 else "outside") for ep in ep_ids), "INFO")

    # x10 size
    total_bytes = float(manifest_df["bytes"].sum())
    total_dur = float(manifest_df["duration_s"].sum())
    add("X10", "total size and GB per hour (info)",
        "%.2f GB total, %.2f GB per hour of video" % (total_bytes / 1e9, total_bytes / 1e9 / (total_dur / 3600)), "INFO")

    hyp = pd.DataFrame(rows)
    save_csv(hyp, REPORTS / "08c_hypothesis_check.csv")
    for r in rows:
        print("%-4s %-5s %s" % (r["id"], r["result"], r["hypothesis"]))
        print("           " + r["measured"])
except Exception as e:
    print("step 10 problem:", type(e).__name__, e)

kept existing 08c_hypothesis_check.csv
X1   PASS  output frame counts equal the plan
           vendorX_2026-10-11_op01_000001 plan 21269 probe 21269 decoded 21269 units 21269; vendorX_2026-10-11_op01_000002 plan 9403 probe 9403 decoded 9403 units 9403; vendorX_2026-10-11_op01_000003 plan 10423 probe 10423 decoded 10423 units 10423; vendorX_2026-10-11_op01_000004 plan 18024 probe 18024 decoded 18024 units 18024
X2   PASS  one resolution 1080x1920 for all four
           1080x1920
X3   PASS  keyframe every 30 frames, no b-frames, units equal frames, sps and pps at every keyframe
           vendorX_2026-10-11_op01_000001 keys every 30: True, b: 0, sps/pps: True, units=frames: True; vendorX_2026-10-11_op01_000002 keys every 30: True, b: 0, sps/pps: True, units=frames: True; vendorX_2026-10-11_op01_000003 keys every 30: True, b: 0, sps/pps: True, units=frames: True; vendorX_2026-10-11_op01_000004 keys every 30: True, b: 0, sps/pps: True, units=frames: True
X4   PASS  no run of identical fr

## 11. Summary

Writes the summary file and zips the 08c reports.

In [12]:
try:
    summary = dict(
        step="08c_exo_video_prep", run_time_utc=run_time_utc, ffmpeg_version=ffmpeg_line,
        source=dict(file=root_rel, sha256=root_sha, root_frames=len(pts), root_start_utc=root_start_utc),
        settings=dict(preset=PRESET, crf=chosen_crf, crf_flag=chosen_flag, crf_reason=chosen_why,
                      target_ssim=TARGET_SSIM, bitrate_cap_mbps=BITRATE_CAP_MBPS),
        episodes=manifest_df.drop(columns=["sha256"]).to_dict("records"),
        hypotheses=hyp.to_dict("records"),
        counts={k: int(v) for k, v in hyp["result"].value_counts().items()},
    )
    sum_path = REPORTS / "08c_summary.json"
    if need(sum_path):
        with open(sum_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2, default=lambda o: o.item() if isinstance(o, np.generic) else str(o))
        print("saved", sum_path.name)
    else:
        print("kept existing", sum_path.name)

    zip_path = REPORTS / "share_08c.zip"
    if need(zip_path):
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
            for f in sorted(REPORTS.glob("08c_*")):
                if f.is_file():
                    z.write(f, arcname=f.name)
        print("saved", zip_path.name)
    else:
        print("kept existing", zip_path.name)

    print()
    print("08c exo video prep")
    print("episodes: %d, frames: %d" % (len(manifest_df), manifest_df["frames"].sum()))
    print("crf %d, preset %s, mean %.2f Mbps" % (chosen_crf, PRESET, manifest_df["mbps"].mean()))
    print("total size: %.2f GB" % (manifest_df["bytes"].sum() / 1e9))
    print("hypotheses:", summary["counts"])
    print("zip:", zip_path)
except Exception as e:
    print("step 11 problem:", type(e).__name__, e)

kept existing 08c_summary.json
kept existing share_08c.zip

08c exo video prep
episodes: 4, frames: 59119
crf 28, preset veryfast, mean 1.57 Mbps
total size: 0.39 GB
hypotheses: {'PASS': 5, 'INFO': 4, 'FLAG': 1}
zip: c:\workstation\3RD YEAR\ego_pipeline\reports\share_08c.zip
